# Chapter 2: Training a Decoder-Only Model (GPT-style)

Chapter 1 built an encoder-decoder transformer for translation-style tasks. Modern LLMs (GPT, Llama, etc.) are **decoder-only**: one stack, every layer's self-attention is causal, no encoder and no cross-attention at all. You'll build that stack from scratch again -- it's not a repeat of Chapter 1, there are real architectural differences -- and this time you'll actually **train** it on real data and watch it generate text.

**What's different from Chapter 1, and why we're rebuilding (not reusing) those parts:**
- Self-attention is causal in *every* layer (Chapter 1's encoder layers were unmasked).
- A single fused `qkv` projection instead of three separate `q_proj`/`k_proj`/`v_proj` layers -- a real GPT-2/nanoGPT implementation detail.
- **Pre-norm** (`x + sublayer(norm(x))`) instead of Chapter 1's post-norm (`norm(x + sublayer(x))`) -- trains more stably at depth, and is what virtually every modern model uses.
- **Learned positional embeddings** instead of Chapter 1's fixed sinusoidal encoding.
- **Weight tying** between the input embedding and output projection.
- A real **training loop** (next-token-prediction loss) and **generation loop** (autoregressive sampling) -- Chapter 1 never trained anything, it only checked forward-pass shapes.

**What we're deliberately *not* re-deriving:** the attention formula itself (`softmax(QK^T/sqrt(d_k))V`). You already built that by hand in Chapter 1 and understand it -- re-deriving identical math has no new learning value. Here we call PyTorch's fused, optimized `F.scaled_dot_product_attention` for the actual attention math, and spend the effort on what's genuinely new instead.

Phases:
1. Tokenizer + data pipeline (TinyStories)
2. Causal self-attention
3. GPT block (pre-norm)
4. Full GPT model
5. Training loop
6. Text generation
7. A real training run

In [ ]:
import math
import time

import tiktoken
import torch
import torch.nn.functional as F
from torch import nn
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 1. Tokenizer + data pipeline

**Tokenizer:** training a BPE tokenizer from scratch is a whole topic on its own and orthogonal to learning the model architecture, so we reuse GPT-2's pretrained BPE tokenizer via `tiktoken` (50257 tokens, includes a special `<|endoftext|>` token). This is the same lazy-but-correct call as using `F.scaled_dot_product_attention` above: don't rebuild what you're not trying to learn right now.

**Dataset:** [TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories) -- short, simple children's stories, ~470M tokens total. We stream a subset (don't download the whole thing) and concatenate every story's tokens into one long 1D array, separated by the `<|endoftext|>` token so the model can learn where stories end.

**The labeling scheme (this is the core idea of decoder-only training):** there's no separate input/output pair like translation. You take a window of `block_size` consecutive tokens as input `x`, and the *same window shifted by one position* as the target `y`. At every position, the model's job is "predict the next token." Concretely, if your token stream is `[5, 9, 2, 7, 1]` and `block_size=4`: `x = [5, 9, 2, 7]`, `y = [9, 2, 7, 1]` -- `y[i]` is always `x[i+1]`.

In [ ]:
enc = tiktoken.get_encoding("gpt2")
EOT = enc.eot_token
VOCAB_SIZE = enc.n_vocab
ds = load_dataset("roneneldan/TinyStories", split="train", streaming=True)


def build_token_buffer(num_stories):
    """Stream `num_stories` from TinyStories, tokenize each, concatenate with EOT separators.
    Returns a 1D torch.LongTensor of token ids."""
    all_ids = []
    for story in ds.take(num_stories):
        all_ids.extend(enc.encode_ordinary(story["text"]) + [EOT])
    return torch.tensor(all_ids, dtype=torch.long)


def get_batch(tokens, batch_size, block_size, device):
    """Sample `batch_size` random windows of length `block_size` from `tokens`.
    Returns (x, y) where y is x shifted one position to the right, both on `device`."""
    starts = torch.randint(0, len(tokens) - block_size - 1, (batch_size,))
    x = torch.stack([tokens[s : s + block_size] for s in starts])
    y = torch.stack([tokens[s + 1 : s + block_size + 1] for s in starts])
    return x.to(device), y.to(device)

In [ ]:
# check
t0 = time.time()
tokens = build_token_buffer(num_stories=20000)
print("token buffer:", tokens.shape, "built in", round(time.time() - t0, 1), "s")
assert tokens.dim() == 1 and tokens.dtype == torch.long
assert (tokens == EOT).sum().item() == 20000, "expected exactly one EOT per story"

x, y = get_batch(tokens, batch_size=4, block_size=16, device=device)
assert x.shape == (4, 16) and y.shape == (4, 16)
assert x.device.type == device and y.device.type == device
# the shift property: y at position i equals x at position i+1
assert torch.equal(x[:, 1:], y[:, :-1])

print("phase 1 ok")

## 2. Causal self-attention

Same underlying math as Chapter 1, two real implementation differences:

**Fused `qkv` projection.** Instead of three separate `nn.Linear(d_model, d_model)` layers for Q/K/V, use one `nn.Linear(d_model, 3 * d_model)` and split its output into three chunks. Mathematically identical to three separate projections (it's literally the same three weight matrices, stacked) -- but one matmul is more efficient than three smaller ones, which is why GPT-2 and nanoGPT do it this way.

**`F.scaled_dot_product_attention(..., is_causal=True)`** instead of manually building a mask. You did the manual masking-and-broadcasting exercise in Chapter 1 already; this built-in does the same thing (and uses a fused, often much faster kernel under the hood) without you having to get the broadcast shape right again.

Everything else -- splitting into heads, running attention, merging heads, `out_proj` -- is exactly what you built in Chapter 1's `MultiHeadAttention`.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        # TODO: self.qkv = nn.Linear(d_model, 3 * d_model); self.out_proj = nn.Linear(d_model, d_model)
        # self.dropout = dropout  (store the float, F.scaled_dot_product_attention takes dropout_p directly)
        # then mark: self.out_proj.RESIDUAL_SCALE_INIT = True
        # (GPT._init_weights in Phase 4 looks for this flag -- see that phase's markdown for why)
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = dropout
        self.out_proj.RESIDUAL_SCALE_INIT = True

    def forward(self, x):
        """x: (batch, seq_len, d_model) -> (batch, seq_len, d_model)"""
        # TODO:
        # 1. q, k, v = self.qkv(x).chunk(3, dim=-1)
        # 2. reshape each to (batch, n_heads, seq_len, d_k) -- same as Chapter 1's _split_heads
        # 3. out = F.scaled_dot_product_attention(q, k, v, is_causal=True,
        #                                          dropout_p=self.dropout if self.training else 0.0)
        # 4. merge heads back to (batch, seq_len, d_model), then self.out_proj
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        batch_size, seq_len, _ = x.size()
        q = q.view(batch_size, seq_len, self.n_heads, self.d_k)
        k = k.view(batch_size, seq_len, self.n_heads, self.d_k)
        v = v.view(batch_size, seq_len, self.n_heads, self.d_k)

        out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0.0)

        out = out.reshape(batch_size, seq_len, -1)
        out = self.out_proj(out)
        return out

In [ ]:
# check
attn = CausalSelfAttention(d_model=16, n_heads=4).to(device)
attn.eval()  # disable dropout so the causal check below isn't polluted by dropout randomness

x_ = torch.randn(2, 5, 16, device=device)
out = attn(x_)
assert out.shape == (2, 5, 16), out.shape
assert getattr(attn.out_proj, "RESIDUAL_SCALE_INIT", False) is True, "out_proj needs the RESIDUAL_SCALE_INIT marker"

# causal check: changing a FUTURE token must not change an EARLIER position's output
x2 = x_.clone()
x2[:, -1] += 100.0
out2 = attn(x2)
assert torch.allclose(out[:, :-1], out2[:, :-1], atol=1e-4), "leaked future info -- masking is wrong"

print("phase 2 ok")

## 3. GPT block (pre-norm)

Chapter 1's `EncoderLayer` was **post-norm**: `x = norm(x + sublayer(x))` -- the residual addition happens first, normalization comes after. That's what the original 2017 paper used, but it has a known problem: as you stack more layers, the residual stream's scale keeps changing before each norm, which makes very deep stacks unstable to train.

Virtually every modern decoder-only model instead uses **pre-norm**: `x = x + sublayer(norm(x))` -- normalize *before* the sublayer, then add the (unnormalized) residual. This keeps the residual stream's scale more consistent layer to layer, which is part of why 96-layer models are trainable at all. It's a one-line reordering, but it's a real, deliberate architectural choice, not a stylistic one.

One more small swap: the feed-forward block uses `GELU` instead of Chapter 1's `ReLU` -- GPT-2's choice, a smoother activation that tends to work slightly better in practice. `d_ff` is conventionally `4 * d_model`.

In [ ]:
class Block(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        # TODO: build the MLP as four named attributes (not nn.Sequential -- GPT._init_weights in
        # Phase 4 needs to find the residual-projection layer by name):
        #   self.mlp_fc   = nn.Linear(d_model, d_ff)
        #   self.mlp_act  = nn.GELU()
        #   self.mlp_proj = nn.Linear(d_ff, d_model)
        #   self.mlp_drop = nn.Dropout(dropout)
        # then mark: self.mlp_proj.RESIDUAL_SCALE_INIT = True
        self.mlp_fc = nn.Linear(d_model, d_ff)
        self.mlp_act = nn.GELU()
        self.mlp_proj = nn.Linear(d_ff, d_model)
        self.mlp_drop = nn.Dropout(dropout)
        self.mlp_proj.RESIDUAL_SCALE_INIT = True

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp_drop(self.mlp_proj(self.mlp_act(self.mlp_fc(self.ln2(x)))))
        return x

In [ ]:
# check
blk = Block(d_model=16, n_heads=4, d_ff=32).to(device)
out = blk(torch.randn(2, 5, 16, device=device))
assert out.shape == (2, 5, 16), out.shape
assert isinstance(blk.mlp_act, nn.GELU), "expected GELU, not ReLU"
assert getattr(blk.mlp_proj, "RESIDUAL_SCALE_INIT", False) is True, "mlp_proj needs the RESIDUAL_SCALE_INIT marker"

print("phase 3 ok")

## 4. Full GPT model

**`wte` and `wpe` -- two separate lookup tables, added together.** `wte` ("word/token embedding") is `nn.Embedding(vocab_size, d_model)`: a table with one learned `d_model`-dimensional row *per vocabulary token*. Given a token id, `wte(idx)` looks up that token's row -- a vector that, after training, captures something like "what this word means." `wpe` ("word *position* embedding") is `nn.Embedding(block_size, d_model)`: a *separate* table with one learned row *per absolute position* (slot 0, slot 1, ..., slot `block_size - 1`), with no idea which token landed there. `wpe(pos)` looks up "what it means to be the 3rd token in the sequence," independent of whether that 3rd token is "cat" or "ran."

`x = wte(idx) + wpe(pos)` **adds** those two vectors elementwise (that's *why* both tables must share the same `d_model` width -- you can't add vectors of different sizes). The result is a single vector per position that encodes both "which token is here" and "where in the sequence this is." Concretely: if "cat" (token id, say, 42) shows up at position 3 in one sequence and position 7 in another, both get the *same* `wte[42]`, but different `wpe[3]` vs `wpe[7]` -- so the combined vector differs depending on where "cat" appeared, which is exactly the information later attention layers need to tell "cat is the subject here" from "cat is the object there."

**This is genuinely different from Chapter 1's positional encoding, not just relabeled.** Chapter 1's `PositionalEncoding` computed a fixed sinusoidal formula and stored it in a `register_buffer` -- it's a constant, never updated by gradients. `wpe` here is an `nn.Embedding`, i.e. a real trainable parameter matrix: it starts as random noise (via the `_init_weights` below) and the optimizer *learns* useful position vectors during training, the same way it learns `wte`. Neither table exists in Chapter 1's `Encoder`/`Decoder`, which only had `embed` (token) + a fixed `pos_enc` add -- `wpe` replaces that fixed formula with something learned.

**Why this caps how long a sequence the model can handle:** `wpe` only has `block_size` rows -- there is no "row for position 200" if `block_size=128`, the same way there's no row in `wte` for a token outside the vocabulary. That's the actual mechanical reason Phase 6's generation loop has to truncate with `idx[:, -model.block_size:]` -- not a style choice, `wpe(pos)` would simply error (or be meaningless) for `pos >= block_size`.

**Weight tying:** `lm_head` (the final `d_model -> vocab_size` projection) and `wte` (the `vocab_size -> d_model` input embedding) are set to literally share the same weight tensor: `self.lm_head.weight = self.wte.weight`. This makes sense because both matrices are really doing the same job in reverse -- mapping between "token identity" and "d_model-dimensional vector" -- and tying them roughly halves the embedding-related parameter count with little to no quality cost. Chapter 1 used separate `src`/`tgt` vocabularies so this didn't apply there.

**Weight initialization (given below, not a TODO -- but read why):** while building this chapter, training a 6-layer version of this model with PyTorch's *default* `nn.Linear`/`nn.Embedding` init produced a loss of **157** on the very first batch, before any training. The math says it should start near `ln(vocab_size) ≈ 10.8` -- a freshly initialized model that's only ever guessing should be roughly as wrong as guessing uniformly at random over the vocabulary, no better, no worse. 157 means the untrained logits were already huge and badly miscalibrated.

The cause: with `n_layers` pre-norm residual blocks, each block adds its output directly onto the residual stream (`x = x + sublayer(...)`), so the residual stream's variance grows roughly with the *number of layers* if every sublayer's output has the same scale. Stack 6 of them with default init and the final logits come out far too large. The fix, standard since GPT-2: initialize every weight from a small `N(0, 0.02)` distribution, and additionally shrink the std of each block's two "writes back to the residual stream" (`out_proj` in attention, `mlp_proj` in the MLP) by `1/sqrt(2 * n_layers)` -- that's exactly what the `RESIDUAL_SCALE_INIT` flags you added in Phases 2 and 3 are for. `GPT._init_weights` below applies this to every matching submodule via `self.apply(...)`.

This is why the check below tests the loss *at initialization*, not just shapes -- a shape check would have passed even with the broken init.

In [ ]:
class GPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, block_size, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.n_layers = n_layers
        self.wte = nn.Embedding(vocab_size, d_model)
        self.wpe = nn.Embedding(block_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(
            [Block(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        # TODO: tie lm_head's weight to wte's weight
        self.lm_head.weight = self.wte.weight
        self.apply(self._init_weights)  # see Phase 4's markdown for why this matters

    def _init_weights(self, module):
        """Given -- see the markdown above for why deep pre-norm stacks need this."""
        if isinstance(module, nn.Linear):
            std = 0.02
            if getattr(module, "RESIDUAL_SCALE_INIT", False):
                std /= math.sqrt(2 * self.n_layers)
            nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx):
        """idx: (batch, seq_len) token ids -> (batch, seq_len, vocab_size) logits"""
        # TODO:
        b, t = idx.shape
        pos = torch.arange(t, device=idx.device)
        x = self.drop(self.wte(idx) + self.wpe(pos))
        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        x = self.lm_head(x)
        # run x through every block in self.blocks, then self.ln_f, then self.lm_head

        return x

In [ ]:
# check
torch.manual_seed(0)
model = GPT(VOCAB_SIZE, d_model=64, n_heads=4, d_ff=256, n_layers=2, block_size=32).to(device)
idx = torch.randint(0, VOCAB_SIZE, (2, 16), device=device)
logits = model(idx)
assert logits.shape == (2, 16, VOCAB_SIZE), logits.shape
assert model.lm_head.weight.data_ptr() == model.wte.weight.data_ptr(), "lm_head and wte must share storage"

# the real check: an untrained model's loss on RANDOM targets should be close to ln(vocab_size) --
# the random-guess baseline. If this is way off (e.g. 50+), the init is broken, same as the bug
# described in the markdown above.
random_targets = torch.randint(0, VOCAB_SIZE, (2, 16), device=device)
init_loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), random_targets.view(-1)).item()
print("loss at init:", round(init_loss, 2), " (expect close to ln(vocab_size) =", round(math.log(VOCAB_SIZE), 2), ")")
assert abs(init_loss - math.log(VOCAB_SIZE)) < 1.0, "logits are miscalibrated at init -- check weight tying/init"

print("phase 4 ok")

## 5. Training loop

This is the genuinely new part Chapter 1 never touched. The objective is **next-token prediction**: given `x`, predict `y` (which, recall, is just `x` shifted by one position). At every one of the `block_size` positions simultaneously, the model outputs a probability distribution over the 50257 possible next tokens, and `F.cross_entropy` scores how far that is from the actual next token.

**What each variable in `train_step` actually is** (shapes assume `batch=B`, `block_size=T`, `vocab_size=V`):

| variable | shape | what it holds |
|---|---|---|
| `x` | `(B, T)` | a batch of input token-id windows |
| `y` | `(B, T)` | the same windows shifted one token left -- the "correct next token" at every position |
| `logits = model(x)` | `(B, T, V)` | for every position, one raw score per vocabulary token (higher = model thinks it's more likely). **Not** probabilities yet -- just unnormalized scores |
| `logits.view(-1, V)` | `(B*T, V)` | flattens the batch and time axes into one long list of `B*T` independent predictions. Cross-entropy doesn't care *which* sequence or position a prediction came from -- it just wants (prediction, correct-answer) pairs |
| `y.view(-1)` | `(B*T,)` | the matching list of correct token ids, one per row above |
| `loss` | scalar | the average cross-entropy over all `B*T` predictions: how surprised the model was by the true next tokens. `ln(V) ≈ 10.8` means "guessing uniformly at random"; lower is better |

**What each line does:**
- `optimizer.zero_grad()` -- clear the gradients left over from the previous step (PyTorch *accumulates* gradients by default, so you must reset them or steps would interfere).
- `loss.backward()` -- backpropagation: compute, for every parameter, the gradient of the loss w.r.t. that parameter (which direction to nudge it to reduce the loss).
- `optimizer.step()` -- AdamW uses those gradients to actually update every parameter.
- `return loss.item()` -- `loss` is a GPU tensor still attached to the computation graph; `.item()` pulls out the plain Python float so we can log/plot it without holding onto graph memory.

**The standard sanity check for "is my training loop wired correctly at all":** take one tiny batch, and train on *only that batch* for many steps. A correctly-wired model+loss+optimizer should be able to nearly memorize a couple of short sequences -- if the loss doesn't drop sharply, something upstream (the loss computation, the optimizer step, a frozen parameter) is broken, before you ever touch the real, much larger dataset.</cell>

In [ ]:
def train_step(model, optimizer, x, y):
    """One training step: forward, loss, backward, optimizer step. Returns the loss as a float."""
    # TODO:
    logits = model(x)
    loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    return loss.item()


In [ ]:
# check -- overfit a tiny batch and confirm the loss collapses
torch.manual_seed(0)
tiny_model = GPT(VOCAB_SIZE, d_model=64, n_heads=4, d_ff=256, n_layers=2, block_size=16).to(device)
opt = torch.optim.AdamW(tiny_model.parameters(), lr=3e-3)
xb, yb = get_batch(tokens, batch_size=2, block_size=16, device=device)

losses = [train_step(tiny_model, opt, xb, yb) for _ in range(300)]
print("overfit loss: first", round(losses[0], 3), "-> last", round(losses[-1], 3))
assert losses[-1] < losses[0] * 0.2, "loss didn't drop enough -- training loop is likely wrong"

print("phase 5 ok")

## 6. Text generation

The model only knows how to predict *one* next token at a time. To generate a whole continuation, you feed it your prompt, sample one token from its predicted distribution, append that token to the sequence, and feed the *whole thing* back in to predict the next one -- repeat. This autoregressive loop is the only way a decoder-only model produces text (Chapter 1 never needed this since it only checked forward-pass shapes, never generated anything).

**What each variable in `generate` is** (shapes: `B` sequences, growing length `L`, `V=vocab_size`):

| variable | shape | what it holds |
|---|---|---|
| `idx` | `(B, L)` | the running sequence so far -- starts as your prompt, grows by one column each loop iteration |
| `idx_cond` | `(B, ≤block_size)` | the last `block_size` tokens of `idx`. The model can't look at more positions than `wpe` has rows (see Phase 4), so we crop |
| `model(idx_cond)` | `(B, L', V)` | logits at *every* position. But for generation we only care about the prediction *after the last token* |
| `logits = model(idx_cond)[:, -1, :]` | `(B, V)` | just the last position's row of scores -- "what comes next after the current end of the sequence" |
| `probs` | `(B, V)` | those scores turned into a real probability distribution by softmax (sums to 1) |
| `next_id` | `(B, 1)` | one token id sampled from `probs`, appended onto `idx` |

**Temperature -- worked example.** `temperature` divides the logits *before* softmax. Say three candidate tokens have logits `[2.0, 1.0, 0.0]`:
- **T = 1.0** (unchanged) -> softmax -> `[0.67, 0.24, 0.09]`.
- **T = 0.5** (logits become `[4, 2, 0]`) -> softmax -> `[0.87, 0.12, 0.02]` -- *sharper*: the top token dominates, output is more confident/repetitive/deterministic.
- **T = 2.0** (logits become `[1, 0.5, 0]`) -> softmax -> `[0.51, 0.31, 0.19]` -- *flatter*: the long-shot tokens get more chance, output is more random/creative (and more likely to go off the rails).

In the limit T → 0 it becomes pure "always pick the single most likely token" (greedy/argmax); large T approaches uniform random. So temperature is the **creativity vs. coherence dial**.

**top_k -- worked example.** `top_k` keeps only the `k` highest-scoring logits and sets all the rest to `-inf` (which become probability 0 after softmax), so you never accidentally sample a bizarre token from the 50000-long tail. With `top_k=50`, even at high temperature, you only ever sample among the 50 most plausible next tokens. `top_k=None` means "consider the whole vocabulary." The line `logits[logits < v[:, [-1]]] = float("-inf")` does exactly this: `v[:, [-1]]` is the `k`-th largest logit (the cutoff), and anything below it is killed.

**`torch.multinomial` vs `argmax`.** `argmax` would always take the single highest-probability token -- deterministic, and in practice repetitive and dull. `torch.multinomial(probs, 1)` instead draws a token *randomly in proportion to* `probs`: a token with 0.6 probability is picked ~60% of the time, one with 0.1 about 10% of the time. That randomness is what makes each generation different and the text feel less robotic. (This is also why generation needs `@torch.no_grad()` and `model.eval()` -- we're only doing inference, no gradients, and dropout should be off.)

Note `idx_cond = idx[:, -model.block_size:]` -- the model only has positional embeddings up to `block_size`, so once the generated sequence grows past that, you must feed it only the most recent `block_size` tokens.</cell>

In [ ]:
@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=1.0, top_k=None):
    """idx: (batch, seq_len) token ids to continue from. Returns (batch, seq_len + max_new_tokens)."""
    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -model.block_size:]
        logits = model(idx_cond)[:, -1, :] / temperature
        if top_k is not None:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = float("-inf")
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)
    # model.train() before returning idx
    model.train()
    return idx

In [ ]:
# check
start = torch.tensor([enc.encode_ordinary("Once upon a time")], device=device)
out = generate(tiny_model, start, max_new_tokens=10)
assert out.shape == (1, start.shape[1] + 10), out.shape
assert out.min().item() >= 0 and out.max().item() < VOCAB_SIZE

print("phase 6 ok")

## 7. A real training run

Everything above used tiny toy sizes to make the checks fast. Now scale up and actually train something: ~17.6M params, 30k stories (~6.6M tokens, well inside the storage budget in `HANDOFF.md`), batch size 32 (kept modest deliberately -- batch 64 at this model size measured ~7.7GB peak VRAM, uncomfortably close to this card's 8GB; batch 32 measured ~4.0GB, comfortable headroom), 2000 steps.

Measured on an RTX 4060 Ti with the weight-init fix from Phase 4: ~135s total (~2.3 min), loss drops from ~10.8 (the random-guess baseline) to ~2.7. Sample output at that point:

> *Once upon a time, there was a little girl named Lily. She liked to play outside in the backyard. One day, she found a beautiful stone in the air.*

...followed by less coherent fragments -- recognizably TinyStories-style (names, simple sentence structure) but far from perfect, which is expected at this scale/step count. Push `n_steps` / `num_stories` up if you want better samples; both are cheap to increase from here, and you have VRAM headroom to raise `batch_size` back toward 64 if you do.

In [ ]:
real_tokens = build_token_buffer(num_stories=30000)

torch.manual_seed(0)
real_model = GPT(VOCAB_SIZE, d_model=256, n_heads=8, d_ff=1024, n_layers=6, block_size=128, dropout=0.1).to(device)
print(f"params: {sum(p.numel() for p in real_model.parameters())/1e6:.1f}M")

opt = torch.optim.AdamW(real_model.parameters(), lr=3e-4)
batch_size, n_steps = 32, 2000

loss_history = []  # record every step's loss so we can plot the training curve below
t0 = time.time()
for step in range(n_steps):
    xb, yb = get_batch(real_tokens, batch_size, real_model.block_size, device)
    loss = train_step(real_model, opt, xb, yb)
    loss_history.append(loss)
    if step % 200 == 0 or step == n_steps - 1:
        print(f"step {step}: loss {loss:.3f}  ({time.time()-t0:.1f}s elapsed)")

### Watching it train

A column of printed numbers is hard to read a trend from. Plotting `loss_history` makes the model's progress obvious at a glance -- the plotting code below is given (it's not the learning goal, and `matplotlib` is just a tool here).

**What a healthy curve looks like, and what to read from it:**
- It should **start near the dashed `ln(V) ≈ 10.8` baseline** (random guessing) and **drop steeply early**, then flatten into a slow decline -- most of the easy wins come fast.
- The raw per-step line is **noisy** (each step is a different random batch, so loss bounces around); the moving average is the real trend. Judge progress by the smoothed line, not any single step.
- If it **plateaus well above where you'd like**, that's the signal to train longer (`n_steps`), feed more data (`num_stories`), or make the model bigger -- not a bug.
- If it **spikes upward and stays there** (diverges), the learning rate is usually too high.

In [ ]:
import matplotlib.pyplot as plt

# Plotting scaffolding is given -- it's not the learning goal here. The point is to SEE the curve.
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(loss_history, linewidth=0.7, alpha=0.4, label="loss (per step, noisy)")

# a running average smooths the per-step noise so the trend is readable
window = 50
if len(loss_history) >= window:
    smoothed = [sum(loss_history[i - window:i]) / window for i in range(window, len(loss_history) + 1)]
    ax.plot(range(window - 1, len(loss_history)), smoothed, color="C3", linewidth=2, label=f"{window}-step moving avg")

ax.axhline(math.log(VOCAB_SIZE), color="gray", linestyle="--", linewidth=1, label=f"random-guess baseline = ln(V) ≈ {math.log(VOCAB_SIZE):.1f}")
ax.set_xlabel("training step")
ax.set_ylabel("cross-entropy loss")
ax.set_title("Training loss over time")
ax.legend()
plt.show()

In [ ]:
start = torch.tensor([enc.encode_ordinary("Once upon a time")], device=device)
out = generate(real_model, start, max_new_tokens=80, temperature=0.8, top_k=50)
print(enc.decode(out[0].tolist()))